# NeuralEvolve on Google Colab

This notebook runs the **GPU-heavy training** of NeuralEvolve on Colab, so your Mac stays free.
It does not copy the code: it clones the repository and runs the same scripts as on the Mac.

**What runs where**

| Step | Where | Why |
|---|---|---|
| Train the supernet | **Colab** (this notebook) | Hours of GPU work |
| Accuracy dataset + accuracy predictor | **Colab** | GPU work, about 1 h |
| RQ5: train 15 networks from scratch | **Colab** | GPU work |
| Latency dataset, latency predictor, MobileNetV2 latency | **Mac** | Needs Core ML and the Apple Neural Engine |
| Search experiments RQ1-RQ4, final search, validate-final | **Mac** | Uses measured Mac latency |
| Train the final front + baseline from scratch | **Colab** | GPU work |
| iPhone latency, analysis notebook | **Mac** | Xcode / your results |

**Results are saved on Google Drive** (`MyDrive/NeuralEvolve/runs`). Every step can resume where it
stopped, so if Colab disconnects, just run the setup cells again and rerun the step.

**Before you start:** *Runtime → Change runtime type → T4 GPU*.

## 1. Setup (run every time you open the notebook)

In [ ]:
# Is there a GPU?
import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
print(torch.cuda.get_device_name(0), "| PyTorch", torch.__version__)

In [ ]:
# Settings
REPO_URL = "https://github.com/VagifAlyv/NeuralEvolve.git"
BRANCH = "main"
DRIVE_DIR = "/content/drive/MyDrive/NeuralEvolve"   # results live here and survive disconnects
CODE_DIR = "/content/NeuralEvolve"                  # code lives here (re-cloned each session)
WORKERS = 2                                         # Colab has 2 CPU cores

from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(f"{DRIVE_DIR}/runs", exist_ok=True)

In [ ]:
# Get the code. Private repo? Add a GitHub token as a Colab secret named GITHUB_TOKEN
# (key icon in the left sidebar) and give this notebook access to it.
import os, subprocess
url = REPO_URL
try:
    from google.colab import userdata
    url = REPO_URL.replace("https://", "https://" + userdata.get("GITHUB_TOKEN") + "@")
except Exception:
    pass   # public repo, or no token set
if os.path.isdir(f"{CODE_DIR}/.git"):
    subprocess.run(["git", "-C", CODE_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, CODE_DIR], check=True)
print(subprocess.run(["git", "-C", CODE_DIR, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# Install what Colab lacks (PyTorch, NumPy, pandas, scikit-learn are already there; Core ML is not needed here)
!pip install -q "pymoo>=0.6.1"

# runs/ -> Google Drive, so checkpoints and CSVs survive a disconnect
runs = f"{CODE_DIR}/runs"
if not os.path.islink(runs):
    os.symlink(f"{DRIVE_DIR}/runs", runs)
%cd {CODE_DIR}
!python genome.py

## 2. Train the supernet

About 50 epochs is a good trade-off (see the README). The script saves a checkpoint after **every epoch**:
if Colab disconnects, run section 1 again and then this cell. It continues from the last epoch.

- Keep `EPOCHS`, `BATCH` and `LR` the **same** across sessions: the learning-rate schedule depends on them.
- `CUDA out of memory`? Use `BATCH, LR = 64, 0.05` and start over (delete `runs/supernet/main*` on Drive).
- Free Colab disconnects after ~90 min without interaction and after ~12 h in total. Keep the tab open.

In [ ]:
EPOCHS = 50
BATCH, LR = 128, 0.1
!python -u supernet.py --epochs {EPOCHS} --batch-size {BATCH} --lr {LR} --workers {WORKERS} --tag main

In [ ]:
# Training log: loss should go down, accuracy (every 5 epochs) up
import pandas as pd
pd.read_csv("runs/supernet/main_log.csv")

## 3. Accuracy dataset and accuracy predictor

Scores 1000 random networks with the supernet (a few seconds each, about 1 h). Resumable: rerun the cell
and finished networks are skipped.

In [ ]:
!python evaluate.py check --workers {WORKERS}

In [ ]:
!python -u evaluate.py collect --n 1000 --workers {WORKERS}

In [ ]:
!python predictors.py accuracy

## 4. Hand over to the Mac

Download the results and unzip them **inside the project folder on your Mac** (they land in `runs/`):

In [ ]:
!cd {DRIVE_DIR} && rm -f /content/for_mac.zip && zip -r /content/for_mac.zip runs/supernet/main.pt runs/supernet/main_log.csv runs/accuracy_dataset_main.csv runs/accuracy_predictor.pt
from google.colab import files
files.download("/content/for_mac.zip")

Then on the Mac, in the project folder:

```bash
unzip -o ~/Downloads/for_mac.zip
.venv/bin/python latency.py collect --n 400
.venv/bin/python latency.py baseline
.venv/bin/python predictors.py latency
.venv/bin/python experiments.py rq1
.venv/bin/python experiments.py rq2 && .venv/bin/python experiments.py rq2-measure
.venv/bin/python experiments.py rq3 && .venv/bin/python experiments.py rq3-post
.venv/bin/python experiments.py rq4
.venv/bin/python experiments.py final && .venv/bin/python experiments.py validate-final
```

While the Mac does that, Colab can already do section 5.

## 5. RQ5: supernet vs. training from scratch

Trains 15 networks (spread over the accuracy range) from scratch, so you can check whether the supernet
ranks networks correctly (Kendall τ). Needs only section 3. Resumable per network.

In [ ]:
!python -u train_standalone.py --sample runs/accuracy_dataset_main.csv --k 15 --epochs 15 --workers {WORKERS} --out runs/rq5/standalone.csv

## 6. Train the final front and the baseline from scratch

Needs `runs/final/front_validated.csv` from the Mac (`experiments.py validate-final`).
Upload it to Google Drive at **`MyDrive/NeuralEvolve/runs/final/front_validated.csv`**, then run:

In [ ]:
import os
assert os.path.exists("runs/final/front_validated.csv"), "Upload front_validated.csv to MyDrive/NeuralEvolve/runs/final/ first"
!python -u train_standalone.py --from-csv runs/final/front_validated.csv --epochs 40 --workers {WORKERS} --out runs/final/front_trained.csv

In [ ]:
!python -u train_standalone.py --baseline --epochs 40 --workers {WORKERS} --out runs/final/baseline_trained.csv

## 7. Download the from-scratch results for the analysis notebook (on the Mac)

In [ ]:
!cd {DRIVE_DIR} && rm -f /content/trained.zip && zip -r /content/trained.zip runs/rq5 runs/final/front_trained.csv runs/final/baseline_trained.csv
from google.colab import files
files.download("/content/trained.zip")